# Evaluating the power control

> How well the learned power control predicts the channel and reaches a target reliability on layouts it has not seen, against C3-WM's predictor and others, what the agents then receive, and the same through a comm-core network.

In [ ]:
#| default_exp comm.evaluation

In [ ]:
#| hide
from nbdev.showdoc import *

Everything is measured on the test layouts, which neither the predictors nor their calibration saw:

1. **The prediction** (`prediction_errors`): the error of `ĝ` in dB (bias and spread), and its coverage (the share of
   transmissions whose gain is at least `ĝ`; a 5% quantile should cover about 95%).
2. **Against other predictors** (`baselines`), fit on the same training layouts: C3-WM's (linear in `|h|²` over the
   last gains, eqs. 24–25), the same fit in dB, a quantile model of the history alone (no geometry), the learned
   model's features fit for the mean gain in dB instead of its 5% quantile (geometry + history mean), and perfect CSI.
   The learned predictor is `LEARNED`: geometry + history, the 5% quantile (comm-core's example 07 names them so).
3. **Reliability against power** (`operating_points`, `frontier`): for each target, the margin and threshold chosen
   on the calibration layouts, and what they give on the test layouts; and the best reliability each predictor
   reaches for each power (a frontier: margins tried on the test layouts themselves, an upper bound). Perfect CSI is
   the reference as defined: exactly `Γ` on every servable transmission, nothing else sent. (Least power for a target
   reliability allows withholding the most expensive transmissions, so a calibrated policy can spend less than perfect
   CSI at a target below 100%.) **Per layout** (`layout_spread`): the reliability in each test layout at those `κ` and
   `τ`, since one margin chosen across layouts gives the target on average over them, not in each one.
4. **What the agents receive** (`received`): the share of the servable messages arriving without a bit error, and the
   share of wrong tokens in those sent, for uncoded BPSK at the SNR each transmission gets.
5. **Through a network** (`compare_network`): every method above, and C3-WM as written, as one block of C3-WM's link
   built from comm-core blocks (`c3wm_link`: fading, power, scheduling, SNR, every bit, tokens, airtime), on the test
   layouts' trajectories; next to the analysis of the same transmissions, which it must agree with. It also gives
   what the analysis can't: the tokens the world model gets (`OOV` when nothing arrived), and the bits sent.
6. **At other target SNRs** (`snr_sweep`): 3–4 again for each `Γ`, the predictors unchanged (they predict the gain,
   which doesn't depend on `Γ`), `κ` and `τ` chosen again on the calibration layouts.

`evaluate_power_control` runs 1–4 and 6; `compare_network` runs 5 (the `evaluate` stage of `mains/comm_power.py` runs both).

In [ ]:
#| export
from __future__ import annotations

import math
import pickle
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
from threadpoolctl import threadpool_limits

from c3jepa_wm.comm.channel_data import ChannelTable, db, episode_fading, table_path
from c3jepa_wm.comm.power_control import (Budget, LearnedChannelInversion, TAUS_DB, c3wm_link, calibrate, calibrate_layouts,
                                          evaluate, fit_predictor, outcome, predict_gain, predict_on)
from c3jepa_wm.comm.sequence import fit_model, predictor_name

## The predictors

In [ ]:
#| export
LEARNED = 'geometry + history quantile (learned)'                   # the saved power control's predictor

def past_gains(data: dict, history: int) -> np.ndarray:
    "The past gains |h|² of each sample (most recent first), from its features."
    return 10 ** (data['X'][:, 3:3 + history] / 10)

REFERENCES = ('large-scale CSI (oracle)', 'conditional quantile (oracle)')
DECOMPOSED = 'learned large-scale + fading statistics'

def rician_quantile(q, nu2, var) -> np.ndarray:
    """The q-quantile of |m + s n|² (n ~ CN(0, 1)) given |m|² = `nu2` and s² = `var`: s²/2 times a noncentral χ² with
    2 degrees of freedom and noncentrality 2|m|²/s² (a Rician power); |m|² where s² = 0. Broadcast over its arguments."""
    from scipy.stats import ncx2
    q, nu2, var = (np.array(a, float) for a in np.broadcast_arrays(q, nu2, var))
    out = nu2.copy()
    on = var > 1e-12
    out[on] = var[on] / 2 * ncx2.ppf(q[on], 2, np.maximum(2 * nu2[on] / var[on], 1e-12))
    return out

def rician_tail(g, nu2, var) -> np.ndarray:
    "P(|m + s n|² ≥ g), as `rician_quantile`'s: 1 or 0 where s² = 0."
    from scipy.stats import ncx2
    g, nu2, var = (np.array(a, float) for a in np.broadcast_arrays(g, nu2, var))
    out = (nu2 >= g).astype(float)
    on = var > 1e-12
    out[on] = ncx2.sf(2 * g[on] / var[on], 2, np.maximum(2 * nu2[on] / var[on], 1e-12))
    return out

class DecomposedPredictor:
    """ĝ = L̂ times the q-quantile of the small-scale fading: the large-scale gain L (dB) and the direct path's share
    K / (K + 1) learned from the features (trees, for their means), the fading's quantile from its Rician model
    (`channel_data._fading`'s) instead of learned. In a possible cell that keeps the step before's fading
    (`possible_still`), the fading is the drift of the last one: |g_{t-1}|² taken as the last gain over L̂, and the
    direct path in phase with it."""

    def __init__(self, large, direct, quantile: float = 0.05):
        self.large, self.direct, self.quantile = large, direct, quantile

    def predict_on(self, data: dict) -> np.ndarray:
        "ĝ (|h|²) on a dataset (`build_dataset`'s, large-scale channel): the lowest over the receiver's possible cells."
        n, k, F = data['X_possible'].shape
        X = data['X_possible'].reshape(n * k, F)
        L = self.large.predict(X)                                                     # dB
        d = np.clip(self.direct.predict(X), 0.0, 0.999)
        rho = np.repeat(data['rho'], k)
        faded, r = np.isfinite(rho), np.nan_to_num(rho)
        still = data['possible_still'].reshape(-1) & faded
        last = 10 ** ((X[:, 3] - L) / 10)                                             # |g_{t-1}|²: the last gain over L̂
        nu2 = np.where(still, ((1 - r) * np.sqrt(d) + r * np.sqrt(last)) ** 2, d)
        var = np.where(still, (1 - d) * (1 - r ** 2), 1 - d)
        fade = np.where(faded, rician_quantile(self.quantile, nu2, var), 1.0)
        v = db(10 ** (L / 10) * fade).reshape(n, k)
        return 10 ** (np.where(data['possible_mask'], v, np.inf).min(1) / 10)

def fit_decomposed(train: dict, quantile: float = 0.05, max_iter: int = 400) -> DecomposedPredictor:
    "`DecomposedPredictor` fit on `train` (`build_dataset`'s on the large-scale channel: its `large_scale` and `direct`)."
    from sklearn.ensemble import HistGradientBoostingRegressor
    large = fit_predictor(train['X'], train['large_scale'], None, max_iter)
    direct = HistGradientBoostingRegressor(max_iter=max_iter, learning_rate=0.05, early_stopping=False,
                                           random_state=0).fit(train['X'], train['direct'])
    return DecomposedPredictor(large, direct, quantile)

LEVELS = np.array([0.002, 0.005, 0.01, 0.02, 0.035, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 0.9])   # outages of the distributions
QUANTILE_LEVELS = np.array([0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 0.7, 0.9])                  # `QuantileSet`'s models

def _candidates(data: dict, predict) -> np.ndarray:
    "`predict` (rows of features -> values) on each sample's candidate receiver cells: (samples, candidates), NaN where none."
    n, k, F = data['X_possible'].shape
    v = np.asarray(predict(data['X_possible'].reshape(n * k, F)), float).reshape(n, k)
    return np.where(data['possible_mask'], v, np.nan)

def _row_cdf(x: np.ndarray, xp: np.ndarray, fp: np.ndarray) -> np.ndarray:
    """Per row, the CDF at `x` (samples,) from its quantiles `xp` (samples, levels; increasing) at the levels `fp`:
    linear in between, `fp[0]` below the lowest, 1 above the highest (both pessimistic for the success at ρ_max)."""
    K, rows = xp.shape[1], np.arange(len(x))
    k = (xp <= x[:, None]).sum(1)
    lo, hi = np.clip(k - 1, 0, K - 1), np.clip(k, 0, K - 1)
    x0, x1 = xp[rows, lo], xp[rows, hi]
    w = np.where(x1 > x0, (x - x0) / np.where(x1 > x0, x1 - x0, 1), 0.0)
    f = fp[lo] + w * (fp[hi] - fp[lo])
    return np.where(k == 0, fp[0], np.where(k == K, 1.0, f))

class MeanSpread:
    """A distribution of the gain (dB) per transmission: the mean predictor's μ, an error-size predictor's σ (trees on
    |Ḡ - μ|), and the shape of the normalized error (Ḡ - μ)/σ on the calibration layouts (`calibrate`); on the receiver's
    candidate cell of least μ."""

    def __init__(self, mean, spread, levels=LEVELS):
        self.mean, self.spread, self.levels, self.errors = mean, spread, np.asarray(levels, float), None

    def _pessimistic(self, data: dict) -> tuple:
        mu = _candidates(data, self.mean.predict)
        sd = _candidates(data, lambda X: np.maximum(self.spread.predict(X), 0.1))
        i = np.nanargmin(mu, axis=1)
        rows = np.arange(len(i))
        return mu[rows, i], sd[rows, i]

    def calibrate(self, cal: dict):
        mu, sd = self._pessimistic(cal)
        self.errors = np.sort((db(cal['gain']) - mu) / sd)
        return self

    def quantiles(self, data: dict) -> np.ndarray:
        "The gain's quantiles (dB) at `levels`: (samples, levels)."
        mu, sd = self._pessimistic(data)
        return mu[:, None] + sd[:, None] * np.quantile(self.errors, self.levels)[None, :]

    def cdf(self, data: dict, g_db: float) -> np.ndarray:
        "P(Ḡ ≤ g_db) per transmission."
        mu, sd = self._pessimistic(data)
        return np.searchsorted(self.errors, (g_db - mu) / sd, side='right') / len(self.errors)

class QuantileSet:
    """A distribution of the gain (dB) per transmission: trees for its quantiles at `levels` (the pinball loss), the
    lowest over the receiver's candidate cells, each shifted on the calibration layouts so that a fraction 1 - q of the
    gains is above it (`calibrate`: conformalized per level), sorted where they cross."""

    def __init__(self, models: dict):
        self.models, self.levels, self.shift = models, np.array(sorted(models)), None

    def _raw(self, data: dict) -> np.ndarray:
        return np.stack([np.nanmin(_candidates(data, self.models[q].predict), axis=1) for q in self.levels], axis=1)

    def calibrate(self, cal: dict):
        err = self._raw(cal) - db(cal['gain'])[:, None]
        self.shift = np.array([np.quantile(err[:, k], 1 - q) for k, q in enumerate(self.levels)])
        return self

    def quantiles(self, data: dict) -> np.ndarray:
        return np.sort(self._raw(data) - self.shift[None, :], axis=1)

    def cdf(self, data: dict, g_db: float) -> np.ndarray:
        q = self.quantiles(data)
        return _row_cdf(np.full(len(q), g_db), q, self.levels)

def fit_distributional(train: dict, mean_model=None, max_iter: int = 400, quantile_levels=QUANTILE_LEVELS) -> dict:
    "name -> the distributional predictors (`MeanSpread`, `QuantileSet`) fit on `train` (to calibrate on the calibration layouts)."
    from sklearn.ensemble import HistGradientBoostingRegressor
    mean = mean_model if mean_model is not None else fit_predictor(train['X'], train['gain'], None, max_iter)
    spread = HistGradientBoostingRegressor(max_iter=max_iter, learning_rate=0.05, early_stopping=False, random_state=0).fit(
        train['X'], np.abs(db(train['gain']) - mean.predict(train['X'])))
    return {DISTRIBUTIONAL[0]: MeanSpread(mean, spread),
            DISTRIBUTIONAL[1]: QuantileSet({float(q): fit_predictor(train['X'], train['gain'], float(q), max_iter)
                                            for q in quantile_levels})}

DISTRIBUTIONAL = ('mean + spread, λ-policy', 'quantiles, λ-policy')

def fit_baselines(train: dict, history: int, quantile: float = 0.05, max_iter: int = 400, extra=(),
                  distributional: bool = True) -> dict:
    """The baselines' models, fit on `train`: C3-WM's `Ψ` (linear in |h|², eq. 24: `psi`), the same fit in dB (`psi_db`),
    a quantile model of the history alone, the past gains and the moves (`history_model`), and the learned model's
    features (geometry + history) fit for the mean gain in dB (`mean_model`); on the large-scale channel, the
    `DecomposedPredictor` (`decomposed`); with `distributional`, `fit_distributional`'s (`distributional`); and the
    predictors of `extra`
    (`extra`: name -> model), each a dict of `comm.sequence.fit_model`'s options (kind, quantile, steps, trees,
    sequence), named by `predictor_name`."""
    from comm_core import fit_gain_predictor
    sequences = [np.r_[p[::-1], g] for p, g in zip(past_gains(train, history), train['gain'])]
    out = dict(psi=fit_gain_predictor(sequences, history=history),
                psi_db=fit_gain_predictor(sequences, history=history, domain='db'),
                history_model=fit_predictor(train['X'][:, 3:], train['gain'], quantile, max_iter),
                mean_model=fit_predictor(train['X'], train['gain'], None, max_iter),
                extra={predictor_name(s.get('kind', 'trees'), s.get('quantile', 0.05), s.get('steps', False)): fit_model(train, **s)
                       for s in extra},
                **({'decomposed': fit_decomposed(train, quantile, max_iter)} if 'large_scale' in train else {}))
    if distributional:
        out['distributional'] = fit_distributional(train, out['mean_model'], max_iter)
    return out

PSI_FLOOR = 1e-30                                                   # eq. 25's projection: 0 would ask for infinite power,
                                                                    # a floor asks for ρ_max at τ = ∞, as the analysis sends it

def baselines(train: dict, history: int, quantile: float = 0.05, max_iter: int = 400, fitted: dict | None = None) -> dict:
    """name -> predict(dataset) -> ĝ: C3-WM's predictor (in |h|², and in dB), a history-only quantile model, the
    geometry + history mean, the `extra` predictors of `fitted`, and perfect CSI; fit on `train` (or `fitted`). On the
    large-scale channel also the `DecomposedPredictor` and two references that know what no predictor can: the true
    large-scale gain L (`REFERENCES[0]`), and L times the `quantile` of the fading given the steps before
    (`REFERENCES[1]`: only the fading's new draw unknown; with κ = 0 every transmission's outage is the quantile, the
    per-transmission problem's solution). The cheapest policy for the reliability on average is `optimal_reference`'s."""
    f = fitted if fitted is not None else fit_baselines(train, history, quantile, max_iter)
    return {
        "C3-WM's (linear in |h|²)": lambda d: np.array([max(f['psi'](p), PSI_FLOOR) for p in past_gains(d, history)]),
        "C3-WM's, in dB": lambda d: np.array([f['psi_db'](p) for p in past_gains(d, history)]),
        'history quantile': lambda d: 10 ** (f['history_model'].predict(d['X'][:, 3:]) / 10),
        'geometry + history mean': lambda d: predict_on(f['mean_model'], d),
        **{name: (lambda d, m=m: predict_on(m, d)) for name, m in f.get('extra', {}).items()},
        **({DECOMPOSED: lambda d: f['decomposed'].predict_on(d)} if 'decomposed' in f else {}),
        **({REFERENCES[0]: lambda d: d['large_scale'].copy(),
            REFERENCES[1]: lambda d: d['large_scale'] * rician_quantile(quantile, d['fading_nu2'], d['fading_var'])}
           if 'large_scale' in train else {}),
        'perfect CSI': lambda d: d['gain'].copy(),
    }

def prediction_errors(estimate: np.ndarray, gain: np.ndarray) -> dict:
    "The error of ĝ in dB (mean: bias, std: spread), and the share of transmissions whose gain is at least ĝ (coverage)."
    error = db(estimate) - db(gain)
    return dict(bias_db=float(error.mean()), spread_db=float(error.std()), coverage=float((gain >= estimate * (1 - 1e-9)).mean()))

## Reliability and power

In [ ]:
#| export
def operating_points(cal: tuple, test: tuple, g_min: float, targets=(0.9, 0.95, 0.99), taus_db=TAUS_DB, slack: float = 0.0) -> dict:
    """For each target: the margin and threshold chosen on the calibration layouts (`cal` = (ĝ, gain)), and the
    reliability, power and withheld share they give on the test layouts (`test` = (ĝ, gain))."""
    out = {}
    for target in targets:
        chosen = calibrate(*cal, g_min, target + slack, taus_db=taus_db)
        out[target] = dict(evaluate(*test, g_min, chosen['margin_db'], chosen['tau_db']), margin_db=chosen['margin_db'],
                           tau_db=chosen['tau_db'], met_on_calibration=chosen['met'])
    return out

def frontier(estimate: np.ndarray, gain: np.ndarray, g_min: float, taus_db=(0.0, math.inf)) -> tuple:
    "(power, reliability): the most reliable outcome for each power over margins and thresholds tried on these transmissions (an upper bound)."
    points = np.concatenate([np.c_[o['power'], o['reliability']] for o in (outcome(estimate, gain, g_min, t) for t in taus_db)])
    points = points[np.argsort(points[:, 0])]
    return points[:, 0], np.maximum.accumulate(points[:, 1])

### Per layout

The calibration gives the target over the test transmissions pooled; a layout where the predictor errs more misses
it. `layout_spread` gives the reliability in each test layout (its servable transmissions reaching `Γ`) at the chosen
`κ` and `τ`: how far they spread, the worst, and the share of layouts below the target.

A layout's reliability is measured over a few episodes, and the transmissions of an episode are correlated (the
agents stay in one part of the layout): a layout at exactly the target measures below it about half the time.
With each transmission's `episode`, `layout_intervals` gives each layout an interval by resampling its episodes
(the bootstrap), and `layout_spread` adds the share of layouts **significantly** below the target (the interval
entirely below it) and the spread between layouts with the sampling noise taken out (`between_std`). If they spread widely, no
fixed margin fixes it, and a margin that adapts within a layout (from the transmitter's own feedback) is the remedy.

In [ ]:
#| export
def layout_intervals(reached, servable, layout, episode, confidence: float = 0.95, boots: int = 1000, seed: int = 0) -> dict:
    """Layout -> (low, high, standard error) of its reliability (the share of its servable transmissions `reached`),
    by the bootstrap over its episodes (resampled whole: within one, transmissions are correlated)."""
    rng = np.random.default_rng(seed)
    out = {}
    for k in np.unique(layout):
        on = (layout == k) & servable
        if not on.any():
            continue
        eps, idx = np.unique(episode[on], return_inverse=True)
        r = np.bincount(idx, weights=reached[on].astype(float), minlength=len(eps))
        n = np.bincount(idx, minlength=len(eps)).astype(float)
        draw = rng.integers(len(eps), size=(boots, len(eps)))
        ratio = r[draw].sum(1) / np.maximum(n[draw].sum(1), 1)
        a = (1 - confidence) / 2
        out[int(k)] = (float(np.quantile(ratio, a)), float(np.quantile(ratio, 1 - a)), float(ratio.std()))
    return out

def layout_spread(estimate, gain, layout, g_min: float, margin_db: float, tau_db: float, target: float | None = None,
                  min_servable: int = 20, episode=None, confidence: float = 0.95) -> dict:
    """The reliability in each layout (`per_layout`: layout -> share of its servable transmissions reaching Γ, `servable`:
    their numbers) at one margin and threshold, and its spread over the layouts with at least `min_servable` (mean, std,
    min, 10th percentile, median; with `target`, the share of layouts below it, and the worst shortfall). With each
    transmission's `episode`: each layout's `interval` (`layout_intervals`), the spread between layouts without the
    sampling noise (`between_std`), and with `target`, the share of layouts `significantly_below` it."""
    layout = np.asarray(layout)
    per, count = {}, {}
    for k in np.unique(layout):
        on = layout == k
        n = int((gain[on] >= g_min).sum())
        if n:
            per[int(k)] = float(outcome(estimate[on], gain[on], g_min, tau_db, [margin_db])['reliability'][0])
            count[int(k)] = n
    r = np.array([per[k] for k in per if count[k] >= min_servable])
    out = dict(per_layout=per, servable=count, layouts=len(r))
    if len(r):
        out.update(mean=float(r.mean()), std=float(r.std()), min=float(r.min()), p10=float(np.percentile(r, 10)),
                   median=float(np.median(r)))
        if target is not None:
            out.update(below=float((r < target).mean()), shortfall=float(max(target - r.min(), 0.0)))
        if episode is not None:
            power, sent = transmissions(estimate, g_min, margin_db, tau_db)
            servable = gain >= g_min
            iv = layout_intervals(sent & (power >= (1 / gain) * (1 - 1e-9)), servable, layout, np.asarray(episode), confidence)
            keep = [k for k in per if count[k] >= min_servable]
            se = np.array([iv[k][2] for k in keep])
            out.update(interval={k: iv[k][:2] for k in keep},
                       between_std=float(math.sqrt(max(r.var() - (se ** 2).mean(), 0.0))))
            if target is not None:
                out['significantly_below'] = float(np.mean([iv[k][1] < target for k in keep]))
    return out

def layout_summary(results: dict) -> str:
    "A text table of `evaluate_power_control`'s `layouts`: per predictor and target, the spread of the per-layout reliability."
    names = [n for n, r in results['predictors'].items() if 'layouts' in r]
    targets = list(results['predictors'][names[0]]['layouts'])
    w = max(map(len, names)) + 2
    lines = [f"reliability per test layout, κ and τ chosen on the calibration layouts "
             f"({results['predictors'][names[0]]['layouts'][targets[0]]['layouts']} layouts)",
             f"{'':{w}s}" + ''.join(f"{f'{100 * t:g}%: pooled / mean ± std (between) / p10 / min / below (significantly)':>72s}"
                                    for t in targets)]
    for name in names:
        row = f"{name:{w}s}"
        for t in targets:
            s, pooled = results['predictors'][name]['layouts'][t], results['predictors'][name]['targets'][t]['reliability']
            row += (f"{100 * pooled:22.1f}% {100 * s['mean']:6.1f} ± {100 * s['std']:4.1f} ({100 * s.get('between_std', math.nan):4.1f})"
                    f" {100 * s['p10']:6.1f} {100 * s['min']:6.1f} {100 * s['below']:5.0f}% ({100 * s.get('significantly_below', math.nan):3.0f}%)  ")
        lines.append(row)
    lines.append('below: the share of the test layouts under the target (significantly: their interval over episodes entirely under it); '
                 'between: the std between layouts without the sampling noise')
    return '\n'.join(lines)

def plot_layout_spread(results: dict, target: float, path=None, ax=None):
    "Each predictor's reliability per test layout (a dot each), at the κ and τ chosen for `target`, against the target."
    import matplotlib.pyplot as plt
    names = [n for n, r in results['predictors'].items() if 'layouts' in r and n != 'perfect CSI']
    if ax is None:
        fig, ax = plt.subplots(figsize=(1.6 * len(names) + 2, 4))
    rng = np.random.default_rng(0)
    for i, name in enumerate(names):
        s = results['predictors'][name]['layouts'][target]
        r = 100 * np.array([s['per_layout'][k] for k in s['per_layout'] if s['servable'][k] >= 1])
        ax.plot(i + rng.uniform(-0.15, 0.15, len(r)), r, 'o', ms=5, alpha=0.7, color='#1f77b4' if name == LEARNED else '#7f7f7f')
        ax.plot([i - 0.3, i + 0.3], [100 * results['predictors'][name]['targets'][target]['reliability']] * 2, 'k-', lw=2)
    ax.axhline(100 * target, color='#d62728', lw=1, ls='--', label=f'target {100 * target:g}%')
    ax.set_xticks(range(len(names)), [n.replace(' (learned)', '\n(learned)').replace(', ', ',\n') for n in names], fontsize=8)
    ax.set(ylabel='servable transmissions reaching Γ (%)', title=f'Per test layout (dots) and pooled (bars), κ and τ for {100 * target:g}%')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=8)
    if path is not None:
        ax.figure.savefig(path, dpi=120, bbox_inches='tight')
    return ax

In [ ]:
show_doc(layout_spread)

## What the agents receive

In [ ]:
#| export
def transmissions(estimate: np.ndarray, g_min: float, margin_db: float, tau_db: float) -> tuple:
    "(power in units of Γσ², sent) of each transmission, as the power control decides."
    rho = 10 ** (margin_db / 10) / np.maximum(estimate, 1e-30)
    sent = rho <= 10 ** (tau_db / 10) / g_min if math.isfinite(tau_db) else np.ones(len(rho), bool)
    return np.where(sent, np.minimum(rho, 1 / g_min), 0.0), sent

def received(estimate, gain, g_min: float, margin_db: float, tau_db: float, snr_target_db: float,
             message_bits: int = 392, bits_per_token: int = 8) -> dict:
    """Uncoded BPSK at the SNR each transmission gets (a bit is wrong with probability Q(√(2 SNR))): the share of the
    servable messages arriving without an error, and the share of wrong tokens in the servable messages sent."""
    power, sent = transmissions(estimate, g_min, margin_db, tau_db)
    snr = 10 ** (snr_target_db / 10) * power * gain
    ber = 0.5 * np.vectorize(math.erfc)(np.sqrt(snr))
    servable = gain >= g_min
    clean = np.where(sent, (1 - ber) ** message_bits, 0.0)[servable].mean()
    wrong = (1 - (1 - ber[sent & servable]) ** bits_per_token).mean() if (sent & servable).any() else float('nan')
    return dict(error_free=float(clean), wrong_tokens=float(wrong))

## Through a network

Each method is one power-control block of C3-WM's link (`c3wm_link`), its threshold `τ` the `Schedule` block's, given
to a comm-core `PipelineBackend` (as in comm-core's example *C3-WM's link, built from blocks*). The methods
(`network_methods`):

| method | power control | κ | τ |
|---|---|---|---|
| perfect CSI | `ChannelInversion` (the true gain) | 0 | 0: nothing unservable sent |
| C3-WM as written | `PredictedChannelInversion` with `Ψ` (linear in `|h|²`, eq. 24) | 0 | 0: C3-WM's `δ` |
| each predictor of the analysis (geometry + history quantile, the learned one; C3-WM's `Ψ` in `|h|²` and in dB; history quantile; geometry + history mean) | `LearnedChannelInversion`, or `PredictedChannelInversion` for `Ψ` | chosen on the calibration layouts for the target | chosen with it |
| the learned one with a CRC | the same | the same | the same |

Before a link has `history` past gains, every predictor asks for `ρ_max` (those steps are left out of the results).

`TableFading` gives the channel of a layout's table (full or pair cache) between the nodes' cells, times a fading
factor per pair that the simulation sets at every step, drawn exactly as `build_dataset` draws it (`episode_fading`,
the same seeds). Every agent sends its tokens to every other at every step (random tokens stand in for MsgNet's). The
network sees the same channels as the analysis, so the two must agree, up to the bits' random errors. Per method
(`network_metrics`):

* **reached:** the share of the servable transmissions whose SNR reaches `Γ`;
* **power:** the mean transmit power, relative to perfect CSI's;
* **withheld:** the share not sent (`δ = 0`): the receiver gets `OOV`;
* **intact:** the share of the servable transmissions arriving with every token right;
* **wrong:** the share of wrong tokens in what arrived (the receiver can't tell them from right ones);
* **garbled:** the share of the *unservable* transmissions arriving with wrong tokens (sent at `ρ_max` although no power
  could reach `Γ`); a CRC turns them into `OOV`;
* **cost:** the bits sent per episode (C3-WM's Fig. 8b), in KB.

In [ ]:
#| export
from comm_core import Fading

class TableFading(Fading):
    """The channel between the nodes' cells from a `ChannelTable` (its `channel`: coherent, or large_scale), times a
    fading factor |g|² per pair of nodes (`fades`, default 1)."""

    def __init__(self, table: ChannelTable, channel: str = 'coherent'):
        self.table, self.channel, self.node_positions, self.fades = table, channel, {}, {}

    def set_positions(self, positions):
        "Where the nodes are now: {node: (x, y)}."
        self.node_positions.update({k: tuple(int(v) for v in p) for k, p in positions.items()})

    def coefficient(self, transmission, link) -> complex:
        a, b = self.node_positions[transmission.sender], self.node_positions[transmission.receiver]
        if a == b:
            return math.inf
        fade = self.fades.get(tuple(sorted((transmission.sender, transmission.receiver))), 1.0)
        return complex(math.sqrt(self.table.gain(a, b, self.channel) * fade))

In [ ]:
#| export
def network_methods(
    saved: dict, # `load_power_control`'s result
    fitted: dict, # `fit_baselines`' result
    comparison: dict | None = None, # `evaluate_power_control`'s result: the κ and τ of each predictor (None: the saved one's only)
    target: float = 0.99, # The target whose κ and τ are used (one of `comparison`'s)
) -> dict:
    """name -> the method: `make(walls, positions, headings)` -> its power control, its `margin_db`, `tau_db`, `crc`, and
    the `predictor` (a name of `estimates`') the analysis computes it with."""
    from comm_core import ChannelInversion, PredictedChannelInversion
    b, L = saved['budget'], saved['history']
    snr, rho = b.snr_target_db, b.max_power_w
    def predicted(psi, margin):
        floored = lambda past: max(psi(past), PSI_FLOOR)                  # as the analysis's (eq. 25)
        return lambda walls, positions, headings: PredictedChannelInversion(floored, L, snr, fallback_power=rho, margin_db=margin)
    def learned(model, margin, geometry=True):
        return lambda walls, positions, headings: LearnedChannelInversion(model, walls, positions, headings, L, snr, margin,
                                                                          rho, geometry)
    makers = {LEARNED: lambda m: learned(saved['model'], m),
              "C3-WM's (linear in |h|²)": lambda m: predicted(fitted['psi'], m),
              "C3-WM's, in dB": lambda m: predicted(fitted['psi_db'], m),
              'history quantile': lambda m: learned(fitted['history_model'], m, geometry=False),
              'geometry + history mean': lambda m: learned(fitted['mean_model'], m),
              **{name: (lambda m, model=model: learned(model, m)) for name, model in fitted.get('extra', {}).items()}}
    method = lambda make, margin, tau, predictor, crc=False: dict(make=make, margin_db=float(margin), tau_db=float(tau),
                                                                  crc=crc, predictor=predictor)
    out = {'perfect CSI': method(lambda walls, positions, headings: ChannelInversion(snr), 0.0, 0.0, 'perfect CSI'),
           'C3-WM as written (Ψ, κ = 0, δ)': method(predicted(fitted['psi'], 0.0), 0.0, 0.0, "C3-WM's (linear in |h|²)")}
    points = ({name: r['targets'][target] for name, r in comparison['predictors'].items()} if comparison is not None
              else {LEARNED: saved['calibration']})
    for name, make in makers.items():
        if name in points:
            p = points[name]
            out[f'{name}, κ, τ'] = method(make(p['margin_db']), p['margin_db'], p['tau_db'], name)
    p = points[LEARNED]
    out[f'{LEARNED}, κ, τ, CRC'] = method(makers[LEARNED](p['margin_db']), p['margin_db'], p['tau_db'], LEARNED, crc=True)
    for name, dist in fitted.get('distributional', {}).items():            # Proposition 1's policy on a predicted distribution
        if comparison is None or target not in comparison.get('distributional', {}).get(name, {}):
            continue
        lam = comparison['distributional'][name][target]['lam']
        make = lambda walls, positions, headings, dist=dist, lam=lam: DistributionalChannelInversion(
            dist, lam, b.g_min, walls, positions, headings, L, snr, rho)
        out[f'{name}, λ'] = dict(method(make, math.nan, math.inf, name),
                                 powers=lambda d, dist=dist, lam=lam: _optimal_choice(_distribution_options(dist, d, b.g_min), lam))
    return out

class DistributionalChannelInversion(LearnedChannelInversion):
    """Proposition 1's policy on a predicted distribution (`MeanSpread`, `QuantileSet`) at `lam`: the power minimizing
    power - λ P(success) over the distribution's quantiles and ρ_max, or none (not sent); ρ_max (`fallback_power`)
    while a link has fewer than `history` past gains."""

    def __init__(self, dist, lam: float, g_min: float, walls, positions, headings, history: int = 4,
                 target_snr_db: float = 10.0, fallback_power: float = math.inf):
        super().__init__(None, walls, positions, headings, history, target_snr_db, 0.0, fallback_power)
        self.dist, self.lam, self.g_min = dist, lam, g_min

    @classmethod
    def from_saved(cls, policies: dict, name: str, target: float, walls, positions, headings):
        "The policy `name` at `target` from `load_lambda_policies`' result, for a layout's `walls` (deployment: τ = ∞)."
        p, b = policies['policies'][name], policies['budget']
        return cls(p['dist'], p['lam'][float(target)], b.g_min, walls, positions, headings, policies['history'],
                   b.snr_target_db, b.max_power_w)

    def allocate(self, gain, noise_power, link=None, message=None):
        past = self._past.setdefault((link.sender, link.receiver), [])
        if len(past) < self.history or len(self._rounds) < 3:
            power, send = self.fallback_power, True
        else:
            X = self._rows(link, past)
            data = {'X_possible': X[None], 'possible_mask': np.ones((1, len(X)), bool)}
            units = float(_optimal_choice(_distribution_options(self.dist, data, self.g_min), self.lam)[0])  # × Γσ²
            power, send = units * 10 ** (self.target_snr_db / 10) * noise_power, units > 0
        past.append((float(gain), self._round))
        return power, send

def save_lambda_policies(path, fitted: dict, comparison: dict, saved: dict):
    """Save `evaluate_power_control`'s λ-policies (its 'distributional'): each predicted distribution as calibrated on
    the calibration layouts, with its λ per target, and `saved`'s history and budget (`DistributionalChannelInversion.from_saved`)."""
    policies = {name: dict(dist=fitted['distributional'][name], lam={float(t): float(o['lam']) for t, o in by_target.items()})
                for name, by_target in comparison.get('distributional', {}).items()}
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(dict(policies=policies, history=saved['history'], budget=asdict(saved['budget'])), f)

def load_lambda_policies(path) -> dict:
    "What `save_lambda_policies` saved (the budget as a `Budget`)."
    with open(path, 'rb') as f:
        out = pickle.load(f)
    out['budget'] = Budget(**out['budget'])
    return out

def network_metrics(rows: list, budget: Budget, tokens: int, cost: list) -> dict:
    "The metrics of a network run from its results' rows: (gain, power, sent, arrived, wrong tokens)."
    g, p, sent, ok = (np.array([r[k] for r in rows], float if k < 2 else bool) for k in range(4))
    wrong = np.array([r[4] if r[4] is not None else 0 for r in rows])
    servable, snr = g >= budget.g_min, p * g / budget.noise_w
    gamma = 10 ** (budget.snr_target_db / 10)
    perfect = np.where(servable, gamma * budget.noise_w / g, 0.0).mean()
    return dict(reached=float((sent & (snr >= gamma * (1 - 1e-9)))[servable].mean()), power=float(p.mean() / perfect),
                withheld=float(1 - sent.mean()), intact=float((ok & (wrong == 0))[servable].mean()),
                wrong=float((wrong[ok] / tokens).mean()) if ok.any() else 0.0,
                garbled=float((ok & (wrong > 0))[~servable].mean()) if (~servable).any() else 0.0,
                cost=float(np.mean(cost)), transmissions=len(rows))

def run_network(
    method: dict, # One of `network_methods`'
    saved: dict, # `load_power_control`'s result
    layout_dir, # The layouts' channels (`table_path`)
    trajectory_dir, # The `trajectories_<seed>.npz`
    seeds, # The layouts
    fading: str | None = None, # As in the data: None, or 'rician'
    fading_correlation: float = 0.9,
    symmetric: bool = True, # Full tables: average the two directions
    message_bits: int = 392, # Per message
    bits_per_token: int = 8, # Per token: a codebook of 2**bits_per_token
    seed: int = 0, # As `build_dataset`'s (the fading's draws)
    channel: str = 'coherent', # As in the data: coherent, or large_scale
    skip: int | None = None, # The steps of each episode not judged (as the data's: the history it was built with); None: the history
) -> dict:
    """`network_metrics` of one method through C3-WM's link as blocks, on the trajectories of `seeds` (after each
    episode's first `skip` steps). The predictors run one transmission at a time, on one thread: a thread pool per
    single-row prediction costs far more than the prediction (hours over a run on a node's 72 cores)."""
    with threadpool_limits(1):
        return _run_network(method, saved, layout_dir, trajectory_dir, seeds, fading, fading_correlation, symmetric,
                            message_bits, bits_per_token, seed, channel, skip)

def _run_network(method, saved, layout_dir, trajectory_dir, seeds, fading, fading_correlation, symmetric, message_bits,
                 bits_per_token, seed, channel, skip) -> dict:
    from comm_core import FullMeshTopology, IdealProtocol, Message, Network, Node, PipelineBackend
    budget, history = saved['budget'], saved['history'] if skip is None else skip
    rng = np.random.default_rng([seed, 1])                       # the payloads and the bits: the same for every method
    tokens = message_bits // bits_per_token
    rows, cost = [], []
    for layout in seeds:
        table = ChannelTable(table_path(layout_dir, layout), symmetric=symmetric)
        z = np.load(Path(trajectory_dir) / f'trajectories_{layout}.npz')
        agents = [f'agent_{i}' for i in range(z['cells'].shape[2])]
        fading_channel = TableFading(table, channel)
        fading_rng = np.random.default_rng([seed, int(layout)])        # as build_dataset's
        heads = {}
        control = method['make'](table.walls, lambda: fading_channel.node_positions, lambda: heads)
        backend = PipelineBackend(c3wm_link(fading_channel, control, budget.max_power_w, method['tau_db'], bits_per_token, method['crc']),
                                  noise_power=budget.noise_w)
        net = Network([Node(a) for a in agents], FullMeshTopology(agents), IdealProtocol(), backend)
        for cells, dirs in zip(z['cells'], z['dirs']):
            net.reset(seed=int(rng.integers(2 ** 31)))
            fading_channel.node_positions.clear()
            fades = episode_fading(cells, fading_rng, fading_correlation, table) if fading == 'rician' else {}
            for t in range(len(cells)):
                if (cells[t, :, 0] < 0).any():                     # an agent reached the goal, or the episode ended
                    break
                backend.set_positions({a: tuple(int(v) for v in c) for a, c in zip(agents, cells[t])})
                fading_channel.fades = {(agents[i], agents[j]): f[t] for (i, j), f in fades.items()}
                heads.update({a: int(d) for a, d in zip(agents, dirs[t])})
                for a in agents:
                    for b in agents:
                        if a != b:
                            net.send(Message(a, b, payload=rng.integers(2 ** bits_per_token, size=tokens)))
                for r in net.step():
                    if t >= max(history, 2):
                        rows.append((10 ** (r.metadata['channel_gain_db'] / 10), r.metadata['tx_power'],
                                     not r.metadata.get('outage', False), r.success, r.metadata.get('wrong_tokens')))
            cost.append(net.metrics.summary().bits_sent / 8 / 1000)
    return network_metrics(rows, budget, tokens, cost)

def analysis(estimate, gain, budget: Budget, margin_db: float, tau_db: float, crc: bool = False, message_bits: int = 392,
             power=None) -> dict:
    """What the analysis expects of a method on these transmissions: reached, power, withheld, intact, garbled (BPSK's
    bit errors); from ĝ, κ and τ, or from each transmission's `power` (units of Γσ², 0: withheld)."""
    if power is None:
        r = evaluate(estimate, gain, budget.g_min, margin_db, tau_db)
        power, sent = transmissions(estimate, budget.g_min, margin_db, tau_db)
    else:
        power, sent = np.asarray(power, float), np.asarray(power) > 0
        r = _judged(power, {'gain': gain}, budget.g_min)
    snr = 10 ** (budget.snr_target_db / 10) * power * gain
    clean = (1 - 0.5 * np.vectorize(math.erfc)(np.sqrt(snr))) ** message_bits    # P(no bit wrong)
    servable = gain >= budget.g_min
    garbled = 0.0 if crc or not (~servable).any() else float(np.where(sent, 1 - clean, 0.0)[~servable].mean())
    return dict(reached=r['reliability'], power=r['power'], withheld=r['withheld'],
                intact=float(np.where(sent, clean, 0.0)[servable].mean()), garbled=garbled)

def compare_network(
    saved: dict, # `load_power_control`'s result
    data: dict, # {'train': ..., 'calibration': ..., 'test': ...}: `build_dataset`'s
    layout_dir, # The layouts' channels (`table_path`)
    trajectory_dir, # The `trajectories_<seed>.npz`
    seeds, # The test layouts to run on (some of `data['test']`'s)
    comparison: dict | None = None, # `evaluate_power_control`'s result (None: the learned predictor and the references only)
    target: float = 0.99, # The target whose κ and τ are used
    fitted: dict | None = None, # `fit_baselines`' result (None: fit on `data['train']`)
    quantile: float = 0.05, # Of the history-only baseline, if fit here
    max_iter: int = 400, # Of the history-only baseline, if fit here
    **network, # `run_network`'s options: fading, fading_correlation, symmetric, message_bits, bits_per_token, seed, channel, skip
) -> dict:
    "name -> the method's κ, τ, CRC, and its `network` metrics next to its `analysis` on the same layouts."
    fitted = fitted if fitted is not None else fit_baselines(data['train'], saved['history'], quantile, max_iter)
    methods = network_methods(saved, fitted, comparison, target)
    predict = {**baselines(data['train'], saved['history'], fitted=fitted),
               LEARNED: lambda d: predict_on(saved['model'], d)}
    on = np.isin(data['test']['layout'], list(seeds))
    test = {k: v[on] for k, v in data['test'].items()}
    out = {}
    for name, m in methods.items():
        out[name] = dict(margin_db=m['margin_db'], tau_db=m['tau_db'], crc=m['crc'],
                         network=run_network(m, saved, layout_dir, trajectory_dir, seeds, **network),
                         analysis=analysis(None if 'powers' in m else predict[m['predictor']](test), test['gain'], saved['budget'],
                                           m['margin_db'], m['tau_db'], m['crc'], network.get('message_bits', 392),
                                           m['powers'](test) if 'powers' in m else None))
    return out

def network_summary(results: dict) -> str:
    "A text table of `compare_network`'s result: per method, the network's metrics, and the analysis's below them."
    label = lambda tau: '∞' if math.isinf(tau) else f'{tau:g}'
    w = max(map(len, results)) + 2
    head = (f"{'':{w}s}{'κ (dB)':>7s}{'τ':>5s}{'reached':>9s}{'power':>9s}{'withheld':>9s}{'intact':>8s}{'wrong':>8s}"
            f"{'garbled':>9s}{'cost':>12s}")
    lines = [head]
    for name, r in results.items():
        n, a = r['network'], r['analysis']
        lines.append(f"{name:{w}s}{r['margin_db']:7.2f}{label(r['tau_db']):>5s}{100 * n['reached']:8.1f}%{n['power']:8.3g}×"
                     f"{100 * n['withheld']:8.1f}%{100 * n['intact']:7.1f}%{100 * n['wrong']:7.2f}%{100 * n['garbled']:8.1f}%"
                     f"{n['cost']:7.2f} KB/ep")
        lines.append(f"{'  (analysis)':{w + 12}s}{100 * a['reached']:8.1f}%{a['power']:8.3g}×{100 * a['withheld']:8.1f}%"
                     f"{100 * a['intact']:7.1f}%{'':8s}{100 * a['garbled']:8.1f}%")
    lines.append("reached: servable transmissions reaching Γ; power: × perfect CSI's; intact: servable ones with every token "
                 "right; wrong: wrong tokens in what arrived; garbled: unservable ones arriving with wrong tokens")
    return '\n'.join(lines)

In [ ]:
show_doc(compare_network)

In [ ]:
show_doc(network_methods)

## All of it

In [ ]:
#| export
def estimates(
    saved: dict, # `load_power_control`'s result
    data: dict, # {'train': ..., 'calibration': ..., 'test': ...}: `build_dataset`'s
    compare: bool = True, # Also C3-WM's predictor (in |h|², and in dB) and the history-only quantile model, fit on 'train'
    quantile: float = 0.05, # Of the history-only baseline
    max_iter: int = 400, # Of the history-only baseline
    fitted: dict | None = None, # `fit_baselines`' result (None: fit here)
) -> dict:
    "name -> (ĝ on the calibration layouts, ĝ on the test layouts): the learned predictor, perfect CSI, and the baselines if `compare`."
    out = {LEARNED: lambda d: predict_on(saved['model'], d)}
    if compare:
        out.update(baselines(data['train'], saved['history'], quantile, max_iter, fitted))
    out['perfect CSI'] = lambda d: d['gain'].copy()
    return {name: (predict(data['calibration']), predict(data['test'])) for name, predict in out.items()}

def _operating(name, cal, test, budget: Budget, targets, taus_db, slack, message_bits, bits_per_token) -> dict:
    "Per target: κ and τ chosen on `cal`, and what they give on `test` (perfect CSI: the reference, not calibrated)."
    if name == 'perfect CSI':                                     # the reference, as defined: exactly Γ, nothing unservable sent
        points = {t: dict(evaluate(*test, budget.g_min, 0.0, 0.0), margin_db=0.0, tau_db=0.0, met_on_calibration=True)
                  for t in targets}
    else:
        points = operating_points(cal, test, budget.g_min, targets, taus_db, slack)
    return {t: dict(p, **received(*test, budget.g_min, p['margin_db'], p['tau_db'], budget.snr_target_db,
                                  message_bits, bits_per_token)) for t, p in points.items()}

def snr_sweep(
    estimated: dict, # `estimates`' result
    data: dict, # As `estimates`'
    budget: Budget, # σ² and ρ_max (its Γ is replaced)
    snr_targets_db=(4, 7, 10, 13, 16), # The Γ to evaluate at
    targets=(0.9, 0.95, 0.99), # Reliability targets (1 - ε)
    taus_db=TAUS_DB, # Thresholds tried in the calibration
    slack: float = 0.0, # Calibrate for target + slack
    message_bits: int = 392, # Per message
    bits_per_token: int = 8, # Per token
) -> dict:
    """Γ -> the weakest servable gain (dB), the share of servable test transmissions, and per predictor and target the
    operating point on the test layouts (as `evaluate_power_control`'s), with κ and τ chosen again for that Γ."""
    cal_gain, test_gain = data['calibration']['gain'], data['test']['gain']
    out = {}
    for snr in snr_targets_db:
        b = replace(budget, snr_target_db=float(snr))
        out[float(snr)] = dict(
            g_min_db=float(db(b.g_min)), servable=float((test_gain >= b.g_min).mean()),
            predictors={name: _operating(name, (c, cal_gain), (t, test_gain), b, targets, taus_db, slack, message_bits,
                                         bits_per_token) for name, (c, t) in estimated.items()})
    return out

OPTIMAL = 'optimal policy (oracle)'
OUTAGES = np.geomspace(1e-5, 0.6, 64)                             # the outages `optimal_reference` chooses from

def _optimal_options(data: dict, g_min: float, outages=OUTAGES) -> tuple:
    """Each transmission's options knowing L and the fading's distribution given the steps before: the power (units of
    Γσ²) for each outage, and ρ_max with its success probability; `inf` where an outage needs more than ρ_max."""
    L, nu2, var = data['large_scale'][:, None], data['fading_nu2'][:, None], data['fading_var'][:, None]
    power = 1 / (L * rician_quantile(outages[None, :], nu2, var))
    success = np.broadcast_to(1 - outages, power.shape)
    power = np.where(power <= 1 / g_min, power, np.inf)
    cap = rician_tail(g_min / L, nu2, var)                          # at ρ_max: P(G ≥ g_min)
    return np.c_[power, np.full(len(L), 1 / g_min)], np.c_[success, cap]

def _optimal_choice(options: tuple, lam: float) -> np.ndarray:
    "Proposition 1's policy at λ: per transmission, the power minimizing power - λ P(success), or 0 (withheld) if none is below 0."
    power, success = options
    cost = power - lam * success
    k = np.argmin(cost, axis=1)
    best = cost[np.arange(len(k)), k]
    return np.where(best < 0, power[np.arange(len(k)), k], 0.0)

def _judged(power: np.ndarray, data: dict, g_min: float, keep=None) -> dict:
    "Reliability (of the servable), power (× perfect CSI's) and withheld share of powers in units of Γσ² (0: withheld)."
    keep = np.ones(len(power), bool) if keep is None else keep
    g, p = data['gain'][keep], power[keep]
    servable = g >= g_min
    reached = (p > 0) & (p * g >= 1 - 1e-9)
    return dict(reliability=float(reached[servable].mean()), power=float(p.mean() / np.where(servable, 1 / g, 0).mean()),
                withheld=float((p == 0).mean()))

def lagrangian_policy(
    cal_options: tuple, test_options: tuple, # `_optimal_choice`'s options (powers, success probabilities) per transmission
    cal: dict, test: dict, # The transmissions they are of (`build_dataset`'s)
    g_min: float, # The weakest servable gain
    targets=(0.9, 0.95, 0.99), # Reliability targets
    slack: float = 0.0, # λ chosen for target + slack
) -> dict:
    """Proposition 1's policy on the options: per transmission the power minimizing power - λ P(success), or none, λ the
    smallest whose reliability on the calibration transmissions reaches `target` + `slack` (`choose_lambda`). Per target:
    λ, and on the test transmissions the reliability, power (× perfect CSI's) and withheld share, the reliability per
    layout (`layout_*`, `comm.deployment.judge`'s), also after a move and while still (`motion`, on the large-scale
    channel)."""
    from c3jepa_wm.comm.deployment import judge                 # (it imports this module)
    parts = (('moved', test['fresh']), ('still', ~test['fresh'])) if 'fresh' in test else ()
    out = {}
    for target in targets:
        lam = choose_lambda(cal_options, cal, g_min, target + slack)
        power = _optimal_choice(test_options, lam)
        layouts = judge(power, power > 0, test['gain'], test['layout'], g_min, target, episode=test.get('episode'))
        out[target] = dict(lam=lam, **_judged(power, test, g_min),
                           **{k: v for k, v in layouts.items() if k.startswith('layout_')},
                           motion={part: dict(share=float(m.mean()), **_judged(power, test, g_min, m)) for part, m in parts if m.any()})
    return out

def choose_lambda(options: tuple, data: dict, g_min: float, reliability: float) -> float:
    "The smallest λ for which `_optimal_choice` on `options` reaches `reliability` on `data`'s servable transmissions (bisection)."
    scale = float(np.mean(1 / data['gain'][data['gain'] >= g_min]))
    lo, hi = -4.0, 8.0                                            # log10 of λ over the mean perfect-CSI power
    for _ in range(48):
        mid = (lo + hi) / 2
        if _judged(_optimal_choice(options, scale * 10 ** mid), data, g_min)['reliability'] >= reliability:
            hi = mid
        else:
            lo = mid
    return scale * 10 ** hi

def lambda_frontier(options: tuple, data: dict, g_min: float, points: int = 121) -> tuple:
    """(power, reliability) of Proposition 1's policy on `options` (`_optimal_options`, `_distribution_options`) for λ
    swept over 12 decades around the mean perfect-CSI power, on `data`'s transmissions: the λ-policy's frontier, as
    `frontier` is a predictor's over κ and τ (judged on the same transmissions, an upper bound)."""
    scale = float(np.mean(1 / data['gain'][data['gain'] >= g_min]))
    judged = [_judged(_optimal_choice(options, scale * 10 ** e), data, g_min) for e in np.linspace(-4, 8, points)]
    points = np.array(sorted((j['power'], j['reliability']) for j in judged))
    return points[:, 0], np.maximum.accumulate(points[:, 1])

def optimal_reference(
    cal: dict, test: dict, # `build_dataset`'s, on the large-scale channel
    g_min: float, # The weakest servable gain
    targets=(0.9, 0.95, 0.99), # Reliability targets
    slack: float = 0.0, # λ chosen for target + slack
    outages=OUTAGES, # The outages each transmission chooses from (and ρ_max)
) -> dict:
    """The cheapest policy for the reliability on average (Proposition 1's), knowing what `REFERENCES[1]` knows:
    `lagrangian_policy` on the true distribution of each gain given the steps before."""
    return lagrangian_policy(_optimal_options(cal, g_min, outages), _optimal_options(test, g_min, outages), cal, test,
                             g_min, targets, slack)

def _distribution_options(dist, data: dict, g_min: float) -> tuple:
    """Each transmission's options from a predicted distribution (`MeanSpread`, `QuantileSet`): the power reaching each of
    its quantiles, with success probability 1 - q (`inf` beyond ρ_max), and ρ_max with its predicted success probability."""
    power = 10 ** (-dist.quantiles(data) / 10)
    success = np.broadcast_to(1 - dist.levels, power.shape)
    power = np.where(power <= 1 / g_min, power, np.inf)
    cap = 1 - dist.cdf(data, float(db(g_min)))
    return np.c_[power, np.full(len(power), 1 / g_min)], np.c_[success, cap]

def _subset(data: dict, keep: np.ndarray) -> dict:
    return {k: v[keep] for k, v in data.items()}

def cross_fitted(dist, cal: dict, g_min: float) -> tuple:
    """The calibration transmissions' options (`_distribution_options`) with `dist` calibrated on the other half of the
    calibration layouts (each half in turn), and those transmissions in the same order; `dist` is left calibrated on all
    of them. Choosing λ on them (`choose_lambda`) avoids judging the distribution on the transmissions it was shaped on."""
    ids = np.unique(cal['layout'])
    if len(ids) < 2:
        dist.calibrate(cal)
        return _distribution_options(dist, cal, g_min), cal
    half = np.isin(cal['layout'], ids[::2])
    options, parts = [], []
    for fit_on, judged_on in ((half, ~half), (~half, half)):
        dist.calibrate(_subset(cal, fit_on))
        options.append(_distribution_options(dist, _subset(cal, judged_on), g_min))
        parts.append(_subset(cal, judged_on))
    dist.calibrate(cal)
    return (tuple(np.concatenate([o[i] for o in options]) for i in range(2)),
            {k: np.concatenate([p[k] for p in parts]) for k in cal})

def distributional_policies(
    fitted: dict, # `fit_baselines`' result, with `distributional`
    cal: dict, test: dict, # `build_dataset`'s
    g_min: float, targets=(0.9, 0.95, 0.99), slack: float = 0.0,
) -> dict:
    """name -> `lagrangian_policy` on each distributional predictor's distributions: λ chosen on the calibration
    transmissions cross-fitted (`cross_fitted`), the test transmissions' with the distribution calibrated on all of them."""
    out = {}
    for name, dist in fitted.get('distributional', {}).items():
        cal_options, cal_order = cross_fitted(dist, cal, g_min)
        out[name] = lagrangian_policy(cal_options, _distribution_options(dist, test, g_min), cal_order, test, g_min,
                                      targets, slack)
    return out

def evaluate_power_control(
    saved: dict, # `load_power_control`'s result
    data: dict, # {'train': ..., 'calibration': ..., 'test': ...}: `build_dataset`'s
    targets=(0.9, 0.95, 0.99), # Reliability targets (1 - ε)
    taus_db=TAUS_DB, # Thresholds tried in the calibration
    slack: float = 0.0, # Calibrate for target + slack
    quantile: float = 0.05, # Of the history-only baseline
    max_iter: int = 400, # Of the history-only baseline
    message_bits: int = 392, # Per message
    bits_per_token: int = 8, # Per token
    compare: bool = True, # Also the baselines (perfect CSI always)
    snr_targets_db=(), # Also `snr_sweep` at these Γ (in 'snr')
    fitted: dict | None = None, # `fit_baselines`' result (None: fit here)
    layout_coverage: float | None = None, # Also κ and τ chosen over layouts (`calibrate_layouts`), for this share of layouts
    layout_confidence: float | None = None, # ... with this probability over the calibration layouts (None: in expectation)
    min_servable: int = 20, # Layouts with fewer servable transmissions: left out of the per-layout figures
) -> dict:
    """Prediction errors, operating points, frontiers and what the agents receive, for the learned predictor and the
    baselines; with `layout_coverage`, also each one's operating points with κ and τ chosen over layouts
    (`layout_targets`); on the large-scale channel, the same κ and τ on the transmissions after a move and while both
    agents stood still apart (`motion`), `optimal_reference`'s policy (`optimal`), and with `fitted`'s distributional
    predictors their λ-policies (`distributional`)."""
    budget = saved['budget']
    estimated = estimates(saved, data, compare, quantile, max_iter, fitted)
    test_gain = data['test']['gain']
    results = {name: dict(prediction=prediction_errors(t, test_gain),
                          targets=_operating(name, (c, data['calibration']['gain']), (t, test_gain), budget, targets,
                                             taus_db, slack, message_bits, bits_per_token),
                          frontier=[a.tolist() for a in frontier(t, test_gain, budget.g_min)])
               for name, (c, t) in estimated.items()}
    for name, (c, t) in estimated.items():                         # the reliability per test layout, at the same κ and τ
        results[name]['layouts'] = {target: layout_spread(t, test_gain, data['test']['layout'], budget.g_min, p['margin_db'],
                                                          p['tau_db'], target, min_servable, data['test'].get('episode'))
                                    for target, p in results[name]['targets'].items()}
    if 'fresh' in data['test']:                                    # after a move (the fading drawn again), and while both stood still
        parts = {'moved': data['test']['fresh'], 'still': ~data['test']['fresh']}
        for name, (c, t) in estimated.items():
            results[name]['motion'] = {
                part: dict(share=float(m.mean()), prediction=prediction_errors(t[m], test_gain[m]),
                           targets={target: evaluate(t[m], test_gain[m], budget.g_min, p['margin_db'], p['tau_db'])
                                    for target, p in results[name]['targets'].items()})
                for part, m in parts.items() if m.any()}
    if layout_coverage is not None:                                # κ and τ for each layout to reach the target, not their pool
        cal = data['calibration']
        for name, (c, t) in estimated.items():
            if name == 'perfect CSI':
                continue
            results[name]['layout_targets'] = {}
            for target in targets:
                ch = calibrate_layouts(c, cal['gain'], cal['layout'], budget.g_min, target, layout_coverage, layout_confidence,
                                       taus_db=taus_db, min_servable=min_servable)
                results[name]['layout_targets'][target] = dict(
                    evaluate(t, test_gain, budget.g_min, ch['margin_db'], ch['tau_db']), margin_db=ch['margin_db'],
                    tau_db=ch['tau_db'], met_on_calibration=ch['met'], calibration_below=ch['below'], allowed=ch['allowed'],
                    calibration_layouts=ch['layouts'],
                    layouts=layout_spread(t, test_gain, data['test']['layout'], budget.g_min, ch['margin_db'], ch['tau_db'],
                                          target, min_servable, data['test'].get('episode')))
    out = dict(servable=float((test_gain >= budget.g_min).mean()), transmissions=len(test_gain), predictors=results,
               layout_coverage=layout_coverage, layout_confidence=layout_confidence)
    if compare and 'large_scale' in data['test']:
        out['optimal'] = optimal_reference(data['calibration'], data['test'], budget.g_min, targets, slack)
    if compare and fitted is not None and fitted.get('distributional'):
        out['distributional'] = distributional_policies(fitted, data['calibration'], data['test'], budget.g_min, targets, slack)
    frontiers = {}                                                  # the λ-policies' frontiers (λ swept on the test transmissions)
    if 'optimal' in out:
        frontiers[OPTIMAL] = lambda_frontier(_optimal_options(data['test'], budget.g_min), data['test'], budget.g_min)
    for name in out.get('distributional', {}):                      # each distribution as calibrated on all the calibration layouts
        frontiers[name] = lambda_frontier(_distribution_options(fitted['distributional'][name], data['test'], budget.g_min),
                                          data['test'], budget.g_min)
    if frontiers:
        out['frontiers'] = {name: [a.tolist() for a in f] for name, f in frontiers.items()}
    if len(snr_targets_db):
        out['snr'] = snr_sweep(estimated, data, budget, snr_targets_db, targets, taus_db, slack, message_bits, bits_per_token)
    return out

def motion_summary(results: dict, target: float) -> str:
    "A text table of `evaluate_power_control`'s `motion`: per predictor, after a move and while still, at one target's κ and τ."
    names = [n for n, r in results['predictors'].items() if 'motion' in r]
    if not names:
        return ''
    parts = list(results['predictors'][names[0]]['motion'])
    w = max(map(len, names)) + 2
    shares = ', '.join(f"{p} {100 * results['predictors'][names[0]]['motion'][p]['share']:.0f}%" for p in parts)
    lines = [f"{100 * target:g}% (κ and τ of all the transmissions), {shares} of the test transmissions",
             f"{'':{w}s}" + ''.join(f"{p + ': spread / reached / power':>34s}" for p in parts)]
    for name in names:
        m = results['predictors'][name]['motion']
        lines.append(f"{name:{w}s}" + ''.join(f"{m[p]['prediction']['spread_db']:13.1f} dB {100 * m[p]['targets'][target]['reliability']:6.1f}%"
                                               f"{m[p]['targets'][target]['power']:7.2f}×   " for p in parts))
    policies = {**({OPTIMAL: results['optimal']} if 'optimal' in results else {}), **results.get('distributional', {})}
    for name, by_target in policies.items():
        if target not in by_target:
            continue
        o = by_target[target]
        lines.append(f"{name:{w}s}" + ''.join(f"{'':16s}{100 * o['motion'][p]['reliability']:6.1f}%{o['motion'][p]['power']:7.2f}×   "
                                              for p in parts if p in o['motion']))
        lines.append(f"{'':4s}all: {100 * o['reliability']:.1f}% reached, {o['power']:.2f}× perfect CSI's power, "
                     f"{100 * o['withheld']:.1f}% withheld")
    return '\n'.join(lines)

def summary(results: dict) -> str:
    "A text table: per predictor, its prediction errors and, per target, the reliability and power on the test layouts."
    names = list(results['predictors'])
    targets = list(results['predictors'][names[0]]['targets'])
    w = max(map(len, names)) + 2
    head = f"{'':{w}s}{'bias':>7s}{'spread':>8s}{'cover':>7s}" + ''.join(f'{f"{100 * t:g}%: reached / power / error-free":>36s}' for t in targets)
    lines = [f"{results['transmissions']} test transmissions, {100 * results['servable']:.1f}% servable", head]
    for name in names:
        r = results['predictors'][name]
        p = r['prediction']
        row = f"{name:{w}s}{p['bias_db']:+6.1f} {p['spread_db']:6.1f} {100 * p['coverage']:6.1f}%"
        for t in targets:
            o = r['targets'][t]
            flag = ' ' if o['met_on_calibration'] else '✗'
            row += f"{100 * o['reliability']:13.1f}% {o['power']:6.1f}× {100 * o['error_free']:6.1f}%{flag}  "
        lines.append(row)
    lines.append('✗: no margin and threshold reach the target on the calibration layouts (the most reliable is used)')
    return '\n'.join(lines)

def guarantee_summary(results: dict) -> str:
    """A text table: per predictor and target, κ and τ chosen over the transmissions pooled and over layouts: the
    reliability (pooled), the power, and the share of test layouts reaching the target (and significantly below it)."""
    names = [n for n, r in results['predictors'].items() if 'layout_targets' in r]
    if not names:
        return ''
    targets = list(results['predictors'][names[0]]['layout_targets'])
    w = max(map(len, names)) + 2
    cov, conf = results['layout_coverage'], results['layout_confidence']
    lines = [f"κ and τ chosen pooled, and over layouts for {100 * cov:g}% of them"
             + (f" with {100 * conf:g}% confidence" if conf else " (in expectation)")
             + ": reached (pooled) / power / test layouts reaching the target / significantly below",
             f"{'':{w}s}" + ''.join(f"{f'{100 * t:g}%  pooled':>36s}{'over layouts':>36s}" for t in targets)]
    for name in names:
        r, row = results['predictors'][name], f"{name:{w}s}"
        for t in targets:
            for o, s in ((r['targets'][t], r['layouts'][t]), (r['layout_targets'][t], r['layout_targets'][t]['layouts'])):
                flag = ' ' if o['met_on_calibration'] else '✗'
                row += (f"{100 * o['reliability']:13.1f}% {o['power']:6.2f}× {100 * (1 - s.get('below', math.nan)):5.0f}% "
                        f"{100 * s.get('significantly_below', math.nan):5.0f}%{flag}")
        lines.append(row)
    lines.append('✗: not met on the calibration layouts (over layouts: more of them miss than allowed, at any κ and τ)')
    return '\n'.join(lines)

def plot_frontiers(results: dict, path=None, ax=None):
    """The frontiers (best reliability for each power) and the calibrated operating points, per predictor (κ and τ)
    and per λ-policy (`results['frontiers']`: λ swept)."""
    import matplotlib.pyplot as plt
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4.5))
    for name, r in results['predictors'].items():
        if name == 'perfect CSI':
            continue
        power, reliability = r['frontier']
        line, = ax.plot(power, 100 * np.asarray(reliability), lw=2.2 if name == LEARNED else 1.3, label=name)
        for o in r['targets'].values():
            ax.plot(o['power'], 100 * o['reliability'], 'o', color=line.get_color(), ms=5)
    policies = {**({OPTIMAL: results['optimal']} if 'optimal' in results else {}), **results.get('distributional', {})}
    for name, (power, reliability) in results.get('frontiers', {}).items():   # the λ-policies: λ swept (dashed), λ as chosen (squares)
        line, = ax.plot(power, 100 * np.asarray(reliability), '--', lw=1.8, label=f'{name} (λ)')
        for o in policies.get(name, {}).values():
            ax.plot(o['power'], 100 * o['reliability'], 's', color=line.get_color(), ms=5)
    ax.plot(1, 100, 'k*', ms=12, label='perfect CSI')
    ax.set(xscale='log', xlim=(0.5, 50), ylim=(80, 100.5), xlabel='mean power, × perfect CSI',
           ylabel='servable transmissions reaching Γ (%)',
           title='Test layouts: frontiers (lines; dashed: λ swept) and calibrated operating points')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    if path is not None:
        ax.figure.savefig(path, dpi=120, bbox_inches='tight')
    return ax

def snr_summary(sweep: dict) -> str:
    "A text table of `snr_sweep`'s result: per Γ and predictor, per target, the reliability, power and error-free share on the test layouts."
    first = next(iter(sweep.values()))
    targets = list(next(iter(first['predictors'].values())))
    w = max(len(n) for r in sweep.values() for n in r['predictors']) + 2
    head = f"{'Γ (dB)':>7s}{'g_min':>8s}{'servable':>10s}  {'':{w}s}" + ''.join(f'{f"{100 * t:g}%: reached / power / error-free":>36s}' for t in targets)
    lines = [head]
    for snr, r in sweep.items():
        for i, (name, points) in enumerate(r['predictors'].items()):
            row = (f"{snr:7g}{r['g_min_db']:7.0f} {100 * r['servable']:8.1f}%  " if i == 0 else ' ' * 27) + f"{name:{w}s}"
            for t in targets:
                o = points[t]
                flag = ' ' if o['met_on_calibration'] else '✗'
                row += f"{100 * o['reliability']:13.1f}% {o['power']:6.1f}× {100 * o['error_free']:6.1f}%{flag}  "
            lines.append(row)
    lines.append('g_min: the weakest servable gain (dB). ✗: no margin and threshold reach the target on the calibration layouts')
    return '\n'.join(lines)

def plot_snr(sweep: dict, path=None):
    "Per target (rows): the reliability, the power and the error-free messages on the test layouts against Γ, per predictor."
    import matplotlib.pyplot as plt
    snrs = list(sweep)
    names = list(sweep[snrs[0]]['predictors'])
    targets = list(sweep[snrs[0]]['predictors'][names[0]])
    fig, axes = plt.subplots(len(targets), 3, figsize=(13, 3.2 * len(targets)), squeeze=False)
    for row, t in zip(axes, targets):
        for name in names:
            points = [sweep[s]['predictors'][name][t] for s in snrs]
            style = dict(lw=2.2 if name == LEARNED else 1.3, marker='o', ms=3, label=name,
                         **(dict(color='k', ls='--') if name == 'perfect CSI' else {}))
            row[0].plot(snrs, [100 * p['reliability'] for p in points], **style)
            row[1].plot(snrs, [p['power'] for p in points], **style)
            row[2].plot(snrs, [100 * p['error_free'] for p in points], **style)
        row[0].axhline(100 * t, color='grey', lw=0.8, ls=':')
        row[0].set(ylabel=f'target {100 * t:g}%\nservable reaching Γ (%)')
        row[1].set(yscale='log', ylabel='mean power, × perfect CSI')
        row[2].set(ylabel='servable messages error-free (%)')
        for ax in row:
            ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set(xlabel='target SNR Γ (dB)')
    axes[0, 0].legend(fontsize=8)
    fig.suptitle('Test layouts, κ and τ chosen again for each Γ', y=1.0)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=120, bbox_inches='tight')
    return axes

## Example

The stand-in layouts of `comm.channel_data`'s example (no ray tracing), with fading: trajectories in 40 layouts, the
learned power control trained on 24, calibrated on 8 for 95%, and everything evaluated on the other 8.

In [ ]:
import json
import tempfile

from c3jepa_wm.comm.channel_data import PairCache, build_dataset, collect_trajectories, trace_visited_pairs, wall_geometry
from c3jepa_wm.comm.power_control import load_power_control, save_power_control

In [ ]:
env_cfg = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 9, 'num_obstacles': 0, 'max_steps': 60}
tmp = Path(tempfile.mkdtemp())

def stand_in(walls):
    "A tracer for `PairCache`: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    def tracer(tx, rxs):
        return [0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20) for d, w in (wall_geometry(walls, tx, rx) for rx in rxs)]
    return tracer

SEEDS = dict(train=range(0, 24), calibration=range(24, 32), test=range(32, 40))
collect_trajectories(env_cfg, range(40), tmp, episodes=5, max_steps=60)
for seed in range(40):
    trace_visited_pairs(env_cfg, tmp, [seed], tmp, tracer=stand_in(PairCache(tmp / 'unused.npz', env_cfg, seed).walls), progress=False)
data = {k: build_dataset(tmp, tmp, s, channel='large_scale', fading='rician') for k, s in SEEDS.items()}

budget = Budget(snr_target_db=10, noise_dbm=-94, max_power_dbm=10)
model = fit_predictor(data['train']['X'], data['train']['gain'], max_iter=100)
chosen = calibrate(predict_gain(model, data['calibration']['X_possible'], data['calibration']['possible_mask']),
                   data['calibration']['gain'], budget.g_min, target=0.95)
save_power_control(tmp / 'power_control.pkl', model, 4, chosen, budget)
saved = load_power_control(tmp / 'power_control.pkl')

fitted = fit_baselines(data['train'], 4, max_iter=100)
results = evaluate_power_control(saved, data, targets=(0.9, 0.95, 0.99), max_iter=100, snr_targets_db=(4, 7, 10, 13, 16), fitted=fitted,
                                 layout_coverage=0.8)
print(summary(results))

In [ ]:
plot_frontiers(results);

In [ ]:
#| hide
# the λ-policies' frontiers: one per λ-policy and the optimal policy, nondecreasing; each calibrated point (λ chosen on the
# calibration layouts) is reachable on the test transmissions, so no frontier point at most its power does much worse
assert set(results['frontiers']) == {OPTIMAL, *results['distributional']}
for name, (power, reliability) in results['frontiers'].items():
    assert np.all(np.diff(power) >= 0) and np.all(np.diff(reliability) >= 0)
    by_target = results['optimal'] if name == OPTIMAL else results['distributional'][name]
    for o in by_target.values():
        best = max(r for p, r in zip(power, reliability) if p <= o['power'] * 1.05)
        assert best >= o['reliability'] - 0.02, (name, o['power'], o['reliability'], best)

In [ ]:
print(layout_summary(results))
print(guarantee_summary(results))
plot_layout_spread(results, 0.95);

In [ ]:
network = compare_network(saved, data, tmp, tmp, SEEDS['test'], comparison=results, target=0.95, fitted=fitted, channel='large_scale', fading='rician')
print(network_summary(network))

At other target SNRs, with the same predictors (`κ` and `τ` chosen again for each `Γ`):

In [ ]:
print(snr_summary(results['snr']))
plot_snr(results['snr']);

A higher `Γ` raises the weakest servable gain `g_min = Γσ²/ρ_max` (fewer transmissions are servable), and makes the
messages arriving at `Γ` cleaner: perfect CSI's error-free share goes from many corrupted messages at low `Γ` to
nearly all clean at high `Γ`. The reliability and power at a `Γ` are those of a budget with the same `g_min`.

With so few calibration layouts, the target is missed by a few points on new layouts: the margin is chosen to just
reach it on the calibration layouts, which say little about the next one. More calibration layouts narrow the gap (in
comm-core's example *Power control on new layouts*, 6 layouts of a 9 × 9 FindGoal landed within 0.7 points of it), and
`slack` (calibrating for a slightly higher target) covers what remains.

In [ ]:
learned, paper = results['predictors'][LEARNED], results['predictors']["C3-WM's (linear in |h|²)"]
learned_q = learned
assert 0.85 < learned['prediction']['coverage'] <= 1                       # a low quantile: it rarely overestimates
assert learned['targets'][0.95]['reliability'] > 0.9
assert results['predictors']['perfect CSI']['targets'][0.95]['power'] <= 1.0001
spread = learned['layouts'][0.95]                                         # per layout: the pooled reliability is their weighted mean
w = np.array([spread['servable'][k] for k in spread['per_layout']])
assert abs(np.average([spread['per_layout'][k] for k in spread['per_layout']], weights=w) - learned['targets'][0.95]['reliability']) < 1e-9
assert spread['min'] <= learned['targets'][0.95]['reliability'] <= max(spread['per_layout'].values())
assert results['predictors']['perfect CSI']['layouts'][0.95]['min'] == 1.0
assert all(lo - 1e-9 <= spread['per_layout'][k] <= hi + 1e-9 for k, (lo, hi) in spread['interval'].items())   # over episodes
assert spread['significantly_below'] <= spread['below'] and 0 <= spread['between_std'] <= spread['std'] + 1e-12
over = learned['layout_targets'][0.95]                                   # over layouts: 8 calibration layouts, 80%: none may miss
assert over['allowed'] == 0 and over['calibration_layouts'] == 8
assert not over['met_on_calibration'] or over['calibration_below'] == 0
assert 'layout_targets' not in results['predictors']['perfect CSI']
for name, r in network.items():                                            # the network is the analysis
    n, a = r['network'], r['analysis']
    assert abs(n['reached'] - a['reached']) < 0.005, name                 # the same channels (a gain at the threshold may round apart)
    assert abs(n['power'] - a['power']) < 0.01 * a['power'], name
    assert abs(n['withheld'] - a['withheld']) < 0.005, name
    assert abs(n['intact'] - a['intact']) < 0.03, name                     # the bits' errors: random
    assert abs(n['garbled'] - a['garbled']) < 0.05, name
learned = network[f'{LEARNED}, κ, τ']
assert abs(learned['network']['reached'] - results['predictors'][LEARNED]['targets'][0.95]['reliability']) < 0.02
perfect = network['perfect CSI']['network']
assert perfect['reached'] == 1.0 and abs(perfect['power'] - 1) < 1e-6 and perfect['garbled'] == 0
crc = network[f'{LEARNED}, κ, τ, CRC']['network']
mean = results['predictors']['geometry + history mean']
assert 'geometry + history mean, κ, τ' in network and mean['prediction']['coverage'] < learned_q['prediction']['coverage']   # a mean covers less than a 5% quantile
assert crc['wrong'] == 0 and crc['garbled'] == 0 and crc['reached'] == learned['network']['reached']
assert network['C3-WM as written (Ψ, κ = 0, δ)']['margin_db'] == 0 and network['C3-WM as written (Ψ, κ = 0, δ)']['tau_db'] == 0
sweep = results['snr']
assert sweep[10.0]['predictors'] == {n: r['targets'] for n, r in results['predictors'].items()}   # at the saved Γ: the same
assert np.all(np.diff([r['servable'] for r in sweep.values()]) <= 0)        # higher Γ: fewer servable transmissions
clean = [r['predictors']['perfect CSI'][0.95]['error_free'] for r in sweep.values()]
assert np.all(np.diff(clean) >= 0) and clean[-1] > 0.99                     # at exactly Γ: cleaner as Γ rises

### What no predictor can know

On the large-scale channel the gain is $G = L\,|g|^2$: the large-scale gain $L$ of the two cells, which the features can
predict, times the small-scale fading, drawn again whenever an agent moves. Three references separate the two:

* **large-scale CSI (oracle)**: $\hat{y} = L$, the true large-scale gain, with its own $\kappa$ and $\tau$;
* **conditional quantile (oracle)**: $L$ times the `quantile` of the fading given the steps before (only its new draw
  unknown): with $\kappa = 0$, every transmission's outage is the quantile, the per-transmission problem's solution;
* **optimal policy (oracle)** (`optimal_reference`): knowing the same, the cheapest policy for the reliability *on
  average*, Proposition 1's: per transmission the power minimizing $\rho - \lambda P(\text{success})$, or none.

It lets the transmissions whose success is dear (deep fades just after a move) fail more often than those whose success
is cheap, so it needs less power than the conditional quantile for the same average. Between the references and the
learned predictors, the **decomposed predictor** (`DecomposedPredictor`) learns $L$ and the direct path's share and
takes the fading's quantile from its Rician model. Each predictor's reliability and power are also reported after a
move and while both agents stood still (`motion`), at the κ and τ chosen for all the transmissions.

In [ ]:
# the references and the decomposed predictor: the gain is L times the fading, of mean 1
print(motion_summary(results, 0.95))
P = results['predictors']
assert {*REFERENCES, DECOMPOSED} <= set(P) and 'decomposed' in fitted
d = data['test']
assert abs((d['gain'] / d['large_scale']).mean() - 1) < 0.1 and d['possible_still'].shape == d['possible_mask'].shape
assert (d['possible_still'] <= d['possible_mask']).all() and np.isnan(d['rho']).sum() == 0
known = P[REFERENCES[1]]['prediction']['coverage']                      # the fading's true 5% quantile: 95% of the gains above it
assert 0.93 < known < 0.97, known
best = results['optimal'][0.95]                                         # Proposition 1's policy: the cheapest for the target on average
assert best['reliability'] > 0.93
assert all(best['power'] <= r['targets'][0.95]['power'] for n, r in P.items() if n != 'perfect CSI')
assert P[DECOMPOSED]['prediction']['coverage'] > 0.8
m = P[LEARNED]['motion']
assert abs(m['moved']['share'] + m['still']['share'] - 1) < 1e-9 and m['still']['share'] > 0
assert np.allclose(rician_quantile(0.05, 0.0, 1.0), -np.log(0.95))      # Rayleigh: 1 - e^{-g}
assert np.allclose(rician_quantile(0.3, 2.0, 0.0), 2.0)                  # no scattered part: the direct path's power
assert np.allclose(rician_tail(rician_quantile([0.05, 0.5], 0.7, 0.3), 0.7, 0.3), [0.95, 0.5])

### Distributional predictors

A single margin on a point prediction gives every transmission the same correction, so it cannot let the outage vary
across transmissions as Proposition 1's policy does. A **distributional predictor** gives each transmission a
distribution of its gain instead: `MeanSpread` (the mean predictor, a predictor of its error's size, and the error's
shape on the calibration layouts) or `QuantileSet` (quantile trees at several levels, each shifted on the calibration
layouts to its exact coverage). `lagrangian_policy` then applies Proposition 1's rule to the predicted distributions,
with λ chosen on the calibration layouts, as `optimal_reference` does to the true ones.

In [ ]:
# the distributions are calibrated on the calibration layouts, and their λ-policies reach the targets on the test ones
for name, d in fitted['distributional'].items():
    q = d.quantiles(data['calibration'])
    coverage = (db(data['calibration']['gain'])[:, None] >= q).mean(0)
    assert np.abs(coverage - (1 - d.levels)).max() < 0.01, name
    for target, o in results['distributional'][name].items():
        assert o['reliability'] > target - 0.02 and o['power'] > 1, (name, target)
assert set(results['distributional']) == set(DISTRIBUTIONAL)
for name in DISTRIBUTIONAL:                                               # through the network as blocks: as the analysis expects
    r = network[f'{name}, λ']
    assert abs(r['network']['reached'] - r['analysis']['reached']) < 0.005 and abs(r['network']['power'] / r['analysis']['power'] - 1) < 0.01, name
    assert 'layout_below' in results['distributional'][name][0.95]
x, xp = np.array([0.0, 1.5, 5.0, -3.0]), np.tile([0.0, 1.0, 2.0], (4, 1))
assert np.allclose(_row_cdf(x, xp, np.array([0.1, 0.5, 0.9])), [0.1, 0.7, 1.0, 0.1])   # linear between, pessimistic outside

In [ ]:
#| hide
# the λ-policies saved for deployment: the same distributions and λ, so the same powers
save_lambda_policies(tmp / 'lambda_policy.pkl', fitted, results, saved)
policies = load_lambda_policies(tmp / 'lambda_policy.pkl')
assert set(policies['policies']) == set(DISTRIBUTIONAL) and policies['budget'] == saved['budget']
for name in DISTRIBUTIONAL:
    p = policies['policies'][name]
    assert p['lam'] == {t: o['lam'] for t, o in results['distributional'][name].items()}
    mine = _optimal_choice(_distribution_options(fitted['distributional'][name], data['test'], saved['budget'].g_min), p['lam'][0.95])
    theirs = _optimal_choice(_distribution_options(p['dist'], data['test'], policies['budget'].g_min), p['lam'][0.95])
    assert np.array_equal(mine, theirs)
    control = DistributionalChannelInversion.from_saved(policies, name, 0.95, np.zeros((15, 15)), lambda: {}, lambda: {})
    assert control.lam == p['lam'][0.95] and control.history == saved['history']